# freeze-requires-grad — worked example 2: Count Trainable vs. Frozen Parameters After Freezing

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `freeze-requires-grad`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

After applying `requires_grad = False` to a subset of a model's parameters, it is good practice to verify the freeze worked by counting trainable and frozen parameter tensors (and their total scalar count). A frozen parameter contributes zero computational cost to the backward pass and zero entries to the optimizer state — this is the main efficiency motivation for partial freezing in transfer learning.

## Worked solution

We write a utility that reports the breakdown of trainable vs. frozen parameters.

**Setup:** A network with layers of known shapes is partially frozen (encoder frozen, head trainable).

**Count tensors:** Use `sum(1 for p in model.parameters() if p.requires_grad)` for trainable count and the negation for frozen count.

**Count scalars:** Use `sum(p.numel() for p in model.parameters() if p.requires_grad)` for trainable scalar count.

**Expected result:** After freezing the encoder (Linear(8,16): 128+16 = 144 params, Linear(16,8): 128+8 = 136 params) and keeping the head (Linear(8,5): 40+5 = 45 params), trainable count should be 45 and frozen count should be 280.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(35)

def param_stats(model):
    trainable_n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_n    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    trainable_t = sum(1 for p in model.parameters() if p.requires_grad)
    frozen_t    = sum(1 for p in model.parameters() if not p.requires_grad)
    return {
        'trainable_scalars': trainable_n,
        'frozen_scalars': frozen_n,
        'trainable_tensors': trainable_t,
        'frozen_tensors': frozen_t,
    }

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Linear(8, 16)   # 8*16 + 16 = 144 scalars
        self.enc2 = nn.Linear(16, 8)   # 16*8 + 8  = 136 scalars
        self.head = nn.Linear(8, 5)    # 8*5  + 5  = 45 scalars
    def forward(self, x):
        return self.head(t.relu(self.enc2(t.relu(self.enc1(x)))))

model = TinyNet()
stats_before = param_stats(model)
print("Before freeze:", stats_before)
assert stats_before['frozen_scalars'] == 0

# Freeze encoder, leave head
for p in model.enc1.parameters(): p.requires_grad = False
for p in model.enc2.parameters(): p.requires_grad = False

stats_after = param_stats(model)
print("After freeze:", stats_after)
assert stats_after['trainable_scalars'] == 45,   f"Got {stats_after['trainable_scalars']}"
assert stats_after['frozen_scalars']    == 280,  f"Got {stats_after['frozen_scalars']}"
assert stats_after['trainable_tensors'] == 2
assert stats_after['frozen_tensors']    == 4
print("Parameter counts verified.")